# Act 3 — Side-by-Side Evaluation Comparison

This notebook calls the local EvalHub API (`localhost:18080`) directly via `requests`.
No MCP. No extra framework. Just HTTP.

**Edit Cell 1 only** — configure your two approaches.  
Everything else is driven by the config you set there.

In [ ]:
# Cell 1 — Configuration
# ─────────────────────────────────────────────────────────────────────────────
# Researchers: edit APPROACH_A and APPROACH_B to compare any two benchmarks
# or collections. Nothing else needs to change.
# ─────────────────────────────────────────────────────────────────────────────
import os

EVALHUB   = os.environ.get('EVALHUB_ENDPOINT', 'http://localhost:18080')
MODEL_URL  = os.environ.get('MODEL_ENDPOINT',  'https://openrouter.ai/api/v1')
MODEL_NAME = os.environ.get('MODEL_NAME',      'liquid/lfm-2.5-1.2b-instruct:free')

APPROACH_A = {
    'name':        'baseline-bbq',
    'description': 'BBQ intersectional bias — established baseline (Parrish et al. 2022)',
    'benchmarks':  [{'id': 'bbq_generate', 'provider_id': 'lm_evaluation_harness'}],
}

APPROACH_B = {
    'name':        'novel-irr-safety',
    'description': 'IRR-validated safety collection — novel contribution (α = 0.81)',
    'collection':  {'id': 'workshop-irr-safety-v1'},
}

print(f'EvalHub : {EVALHUB}')
print(f'Model   : {MODEL_NAME} @ {MODEL_URL}')

In [ ]:
# Cell 2 — Verify EvalHub is healthy and collections exist
import requests, json

health = requests.get(f'{EVALHUB}/api/v1/health', timeout=5).json()
print(f'EvalHub status : {health["status"]}')

collections = requests.get(f'{EVALHUB}/api/v1/evaluations/collections', timeout=5).json()
col_ids = [c.get('id') or c.get('name') for c in (collections if isinstance(collections, list) else collections.get('items', []))]
print(f'Available collections: {col_ids}')

cid = APPROACH_B['collection']['id']
assert cid in col_ids, f"Collection '{cid}' not found. Run: cd 03-irr-local && python3 scripts/register_benchmark.py"
print(f"\nCollection '{cid}' ✓")

In [ ]:
# Cell 3 — Submit both approaches to EvalHub

def resolve_collection_id(name):
    """Look up a collection UUID by name.
    UUIDs are ephemeral in local-mode SQLite — resolve by name each run."""
    r = requests.get(f"{EVALHUB}/api/v1/evaluations/collections", timeout=10)
    items = r.json() if isinstance(r.json(), list) else r.json().get("items", [])
    for col in items:
        if col.get("name") == name:
            return col["resource"]["id"]
    raise ValueError(f"Collection '{name}' not found. Run: cd 03-irr-local && python3 scripts/register_benchmark.py")

def submit(approach):
    payload = {
        "name":  approach["name"],
        "model": {"url": MODEL_URL, "name": MODEL_NAME},
    }
    if "benchmarks" in approach:
        payload["benchmarks"] = approach["benchmarks"]
    if "collection" in approach:
        col_uuid = resolve_collection_id(approach["collection"]["id"])
        payload["collection"] = {"id": col_uuid}
    if "description" in approach:
        payload["description"] = approach["description"]
    r = requests.post(f"{EVALHUB}/api/v1/evaluations/jobs", json=payload, timeout=15)
    r.raise_for_status()
    job_id = r.json()["resource"]["id"]
    print(f"  Submitted '{approach[chr(39)+"name"+chr(39)]}' → {job_id}")
    return job_id

print("Submitting evaluations:")
job_id_a = submit(APPROACH_A)
job_id_b = submit(APPROACH_B)

In [ ]:
# Cell 4 — Poll until both jobs complete
import time

POLL_INTERVAL = 5
TIMEOUT = 600

def poll(job_id):
    r = requests.get(f'{EVALHUB}/api/v1/evaluations/jobs/{job_id}', timeout=10)
    r.raise_for_status()
    return r.json()

job_ids = [job_id_a, job_id_b]
start = time.time()
while time.time() - start < TIMEOUT:
    statuses = {jid: poll(jid) for jid in job_ids}
    n_done = sum(1 for j in statuses.values()
                 if j.get('status', {}).get('state', '') in ('completed', 'failed'))
    elapsed = int(time.time() - start)
    print(f'  {n_done}/{len(job_ids)} done  ({elapsed}s)', end='\r')
    if n_done == len(job_ids):
        print(f'\n  Both jobs finished in {elapsed}s.')
        break
    time.sleep(POLL_INTERVAL)
else:
    raise TimeoutError(f'Jobs did not complete within {TIMEOUT}s.')

In [ ]:
# Cell 5 — Extract metrics from completed jobs

def extract_metrics(job):
    metrics = {}
    for bench in job.get('results', {}).get('benchmarks', []):
        metrics.update(bench.get('metrics', {}))
    return metrics

def primary_score(metrics):
    for key in ('acc', 'exact_match', 'score'):
        if key in metrics and isinstance(metrics[key], (int, float)):
            return float(metrics[key])
    vals = [v for v in metrics.values() if isinstance(v, (int, float))]
    return sum(vals) / len(vals) if vals else None

metrics_a = extract_metrics(statuses[job_id_a])
metrics_b = extract_metrics(statuses[job_id_b])
score_a   = primary_score(metrics_a)
score_b   = primary_score(metrics_b)

print(f'Approach A — {APPROACH_A["name"]}')
for k, v in {k: v for k, v in metrics_a.items() if 'stderr' not in k}.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

print(f'\nApproach B — {APPROACH_B["name"]}')
for k, v in {k: v for k, v in metrics_b.items() if 'stderr' not in k}.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

In [ ]:
# Cell 6 — Plot comparison
import pathlib
import matplotlib.pyplot as plt

RESULTS_DIR = pathlib.Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

approaches  = [APPROACH_A, APPROACH_B]
scores      = [score_a, score_b]
all_metrics = [metrics_a, metrics_b]
colours     = ['#7bc8f6', '#50fa7b']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f1a')

# Bar chart: primary score
ax = axes[0]
labels = [a['name'] for a in approaches]
values = [s if s is not None else 0.0 for s in scores]
bars   = ax.bar(labels, values, color=colours, width=0.45)
ax.set_ylabel('Primary Score', fontsize=12, color='#e8e8e8')
ax.set_ylim(0, 1.1)
ax.set_title('Overall Score Comparison', fontsize=13, pad=12, color='#f5a623')
ax.set_facecolor('#0f0f1a'); ax.tick_params(colors='#e8e8e8', labelsize=9)
ax.spines[:].set_color('#444')
for bar, score in zip(bars, scores):
    lbl = f'{score:.3f}' if score is not None else 'N/A'
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03, lbl,
            ha='center', fontsize=13, fontweight='bold', color='#e8e8e8')

# Grouped bar chart: per-metric detail
ax2 = axes[1]
common = sorted(set(metrics_a) & set(metrics_b) - {'acc_stderr'})
x, w  = range(len(common)), 0.35
ax2.bar([i - w/2 for i in x], [metrics_a.get(k, 0) for k in common],
        width=w, label=APPROACH_A['name'], color='#7bc8f6')
ax2.bar([i + w/2 for i in x], [metrics_b.get(k, 0) for k in common],
        width=w, label=APPROACH_B['name'], color='#50fa7b')
ax2.set_xticks(list(x))
ax2.set_xticklabels(common, rotation=30, ha='right', fontsize=8, color='#e8e8e8')
ax2.set_ylabel('Metric Value', color='#e8e8e8')
ax2.set_title('Per-Metric Detail', fontsize=13, pad=12, color='#f5a623')
ax2.set_facecolor('#0f0f1a'); ax2.tick_params(colors='#e8e8e8')
ax2.spines[:].set_color('#444')
ax2.legend(fontsize=8, facecolor='#1a1a2e', labelcolor='#e8e8e8')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'comparison_plot.png', dpi=150, facecolor=fig.get_facecolor())
plt.show()
print(f'Plot saved → {RESULTS_DIR / "comparison_plot.png"}')

In [ ]:
# Cell 7 — Export structured report
import json, pathlib

report = {
    'evalhub': EVALHUB,
    'model': {'url': MODEL_URL, 'name': MODEL_NAME},
    'approaches': [
        {
            'name':          a['name'],
            'description':   a.get('description', ''),
            'primary_score': s,
            'metrics':       m,
            'job_id':        jid,
        }
        for a, s, m, jid in zip(approaches, scores, all_metrics, [job_id_a, job_id_b])
    ],
}

report_path = RESULTS_DIR / 'comparison_report.json'
report_path.write_text(json.dumps(report, indent=2))
print(f'Report saved → {report_path}')
print('\nAct 3 complete. Artifacts in results/:')
print('  comparison_plot.png    — figure for paper / slide deck')
print('  comparison_report.json — data for supplementary materials')